# 입도 관광객 데이터 파이프라인 통합 실행

입도 관광객 CSV를 정제한 뒤 Parquet과 프로파일 메타데이터로 저장하고 결과를 검증합니다.

In [1]:
from pathlib import Path
import json

import pandas as pd

# 직접 만든 클래스 임포트
from src.tour_data_etl import TourDataETL
from src.data_profiler import DataProfiler

input_path = Path("data") / "입도 관광객 추이(일별).csv"
parquet_output_path = Path("outputs") / "jeju_arrival_daily.parquet"
metadata_output_path = Path("outputs") / "jeju_arrival_daily_metadata.json"

print("입력 파일:", input_path)
print("입력 파일 존재 여부:", input_path.exists())

입력 파일: data\입도 관광객 추이(일별).csv
입력 파일 존재 여부: True


In [ ]:
etl = TourDataETL()
profiler = DataProfiler()

# 원본 데이터 로드 - csv 파일을 pandas DataFrame으로 로드
raw_df = etl.load_data(input_path)

print("원본 크기:", raw_df.shape)
print("원본 컬럼:", raw_df.columns.tolist())
display(raw_df.head())

원본 크기: (4993, 4)
원본 컬럼: ['기준일자(YYYYMMDD)', '내국인방문객수', '외국인방문객수', '전체방문객수']


,기준일자(YYYYMMDD),내국인방문객수,외국인방문객수,전체방문객수
0,20260902,28287.0,5916.0,34203.0
1,20260901,26392.0,11800.0,38192.0
2,20260831,28022.0,5867.0,33889.0
3,20260830,30354.0,10069.0,40423.0
4,20260829,31185.0,7886.0,39071.0


In [ ]:
# 입도 데이터에 필요한 컬럼명 및 자료형 변환

# 컬럼명 변경
arrival_df = raw_df.rename(
    columns={
        "기준일자(YYYYMMDD)": "date",
        "내국인방문객수": "domestic",
        "외국인방문객수": "foreign",
        "전체방문객수": "total",
    }
)

# 날짜 자료형 변환
arrival_df["date"] = pd.to_datetime(
    arrival_df["date"].astype(str), #문자열로 변환
    format="%Y%m%d", # 날짜 형식 지정
    errors="coerce", # 변환 실패 시 NaT로 처리
)

# 수치형 컬럼 변환
for column in ["domestic", "foreign", "total"]:
    # 컬럼을 숫자형으로 변환
    arrival_df[column] = pd.to_numeric(
        arrival_df[column],
        errors="coerce",
    )

# 기존 transform 적용
clean_df = etl.transform(arrival_df)
# 날짜 기준으로 정렬 후 인덱스 초기화
clean_df = clean_df.sort_values("date").reset_index(drop=True)

print("변환 결과 크기:", clean_df.shape)
print("변환 컬럼:", clean_df.columns.tolist())
print("날짜 범위:", clean_df["date"].min(), "~", clean_df["date"].max())
print("자료형:")
print(clean_df.dtypes)
display(clean_df.head())

변환 결과 크기: (4993, 4)
변환 컬럼: ['date', 'domestic', 'foreign', 'total']
날짜 범위: 2013-01-01 00:00:00 ~ 2026-09-02 00:00:00
자료형:
date        datetime64[ns]
domestic           float64
foreign            float64
total              float64
dtype: object


,date,domestic,foreign,total
0,2013-01-01,NaN,NaN,17010.0
1,2013-01-02,NaN,NaN,22121.0
2,2013-01-03,NaN,NaN,22305.0
3,2013-01-04,NaN,NaN,20604.0
4,2013-01-05,NaN,NaN,20286.0


In [ ]:
# 데이터 품질 확인
# 결측값 확인
print("컬럼별 결측값:")
print(clean_df.isna().sum())

# 중복 행 및 날짜 확인
print("\n중복 행 수:", int(clean_df.duplicated().sum()))
print("중복 날짜 수:", int(clean_df["date"].duplicated().sum()))

# 내국인+외국인과 전체 합계 비교
complete_components = clean_df[["domestic", "foreign", "total"]].notna().all(axis=1)
# 내국인+외국인과 전체 합계가 다른 행 확인
sum_mismatch = (
    clean_df.loc[complete_components, "domestic"]
    + clean_df.loc[complete_components, "foreign"]
    != clean_df.loc[complete_components, "total"]
)

print("합계 비교 가능 행 수:", int(complete_components.sum()))
print("내국인+외국인과 전체가 다른 행 수:", int(sum_mismatch.sum()))

컬럼별 결측값:
date          0
domestic    969
foreign     969
total        30
dtype: int64

중복 행 수: 0
중복 날짜 수: 0
합계 비교 가능 행 수: 4024
내국인+외국인과 전체가 다른 행 수: 1


In [ ]:
# Parquet 저장, 프로파일링 및 태그 생성
saved_parquet_path = etl.save_to_parquet(
    clean_df,
    parquet_output_path,
)

# 프로파일링 및 태그 생성
stats = profiler.profile_data(clean_df)
# 태그 생성
tags = profiler.generate_tags(stats)

print("Parquet 저장 완료:", saved_parquet_path)
print("프로파일 결과:")
print(json.dumps(stats, ensure_ascii=False, indent=2))
print("생성된 태그:", tags)

100%|██████████| 4/4 [00:00<00:00, 183.42it/s]


Parquet 저장 완료: outputs\jeju_arrival_daily.parquet
프로파일 결과:
{
  "row_count": 4993,
  "column_count": 4,
  "column_names": [
    "date",
    "domestic",
    "foreign",
    "total"
  ],
  "data_types": {
    "date": "DateTime",
    "domestic": "Numeric",
    "foreign": "Numeric",
    "total": "Numeric"
  },
  "missing_count": {
    "date": 0,
    "domestic": 969,
    "foreign": 969,
    "total": 30
  },
  "missing_rate": {
    "date": 0.0,
    "domestic": 0.19407170038053276,
    "foreign": 0.19407170038053276,
    "total": 0.006008411776487082
  },
  "overall_missing_rate": 0.09853795313438814,
  "duplicate_count": 0,
  "numeric_columns": [
    "domestic",
    "foreign",
    "total"
  ],
  "numeric_column_count": 3,
  "datetime_columns": [
    "date"
  ],
  "has_datetime": true,
  "categorical_columns": [],
  "categorical_column_count": 0
}
생성된 태그: ['time-series', 'multivariate']


In [ ]:
# 메타데이터 JSON 저장
# 데이터셋 설명
dataset_info = {
    "id": "JEJU_ARRIVAL_DAILY_01",
    "name": "제주 입도 관광객 추이 일별 데이터",
    "source_file": input_path.name,
}
# 메타데이터 저장 
saved_metadata_path = profiler.export_metadata(
    dataset_info=dataset_info,
    profile_stats=stats,
    tags=tags,
    out_meta_path=metadata_output_path,
)

print("메타데이터 저장 완료:", saved_metadata_path)

메타데이터 저장 완료: outputs\jeju_arrival_daily_metadata.json


In [ ]:
# 저장 결과 재로드 & 확인 및 검증
parquet_check = pd.read_parquet(saved_parquet_path)

with saved_metadata_path.open("r", encoding="utf-8") as file:
    metadata_check = json.load(file)

# 검증
assert parquet_check.shape == clean_df.shape
assert parquet_check.columns.tolist() == clean_df.columns.tolist()
assert len(parquet_check) == 4993
assert parquet_check["date"].notna().all()
assert int(parquet_check.duplicated().sum()) == 0
assert metadata_check["dataset_info"] == dataset_info
assert metadata_check["profile_stats"] == stats
assert metadata_check["tags"] == tags

print("Parquet 재로드 결과:", parquet_check.shape)
print("메타데이터 태그:", metadata_check["tags"])
print("입도 관광객 파이프라인 검증 통과")

Parquet 재로드 결과: (4993, 4)
메타데이터 태그: ['time-series', 'multivariate']
입도 관광객 파이프라인 검증 통과
